In [ ]:
import numpy as np
import pickle
import os
from belief_dynamics_learning.world2d import World2D
from belief_dynamics_learning.dpf import DPF

%load_ext autoreload
%autoreload 2

In [ ]:
num_particles = 100
dt=0.1

# object properties 
dim_object = np.array([0.067, 0.14])
qo_gt = np.array([0, 1, 0]) # ground truth object pose
sigma_pos = 0.06 # standard deviation of position noise

# robot properties
qr_0 = np.array([0, 0.7])
r_robot = 0.05


In [ ]:
world = World2D(num_particles, qo_gt, dim_object, sigma_pos, 
                qr_0, r_robot, dt)
world.plot_belief()

In [ ]:
world.sample_gt_object_pose()
world.sample_robot_pose()
q_r_hist, o_hist, a_hist = world.rollout(100, world.qo_gt.copy())

# test print
# print("Robot pose history:\n")
# print(q_r_hist)
# print("Observation history:\n")
# print(o_hist)
# print("Action history:\n")
# print(a_hist)

# only keep bits of sequence around positive contact measurements

contact_idx = np.where(o_hist==1)[0]
if len(contact_idx) == 0:
    print('No contact detected')
else:
    min_idx = np.min(contact_idx)
    max_idx = np.max(contact_idx)
    buffer_lb = np.min([min_idx, 10])
    buffer_ub = np.min([len(o_hist)-max_idx, 10])
    keep_idx = [min_idx-buffer_lb, max_idx+buffer_ub]
    print('Keeping indices:', keep_idx)
    q_r_hist = q_r_hist[keep_idx[0]:keep_idx[1]]
    o_hist = o_hist[keep_idx[0]:keep_idx[1]]
    world.plot_rollout(q_r_hist, o_hist)

In [ ]:
# initialise hyperparameters for the differentiable particle filter (DPF)
propose_ratio = 0.7
proposer_keep_ratio = 0.15
min_obs_likelihood = 0.004

dpf = DPF(propose_ratio, proposer_keep_ratio, min_obs_likelihood, world)

In [ ]:
# find file path to the training data
root = os.getcwd()
save_path = os.path.join(root, '../data/data.pkl')

# ensure the training data file exists before attempting to load it
if os.path.exists(save_path):
    with open(save_path, 'rb') as f:
        data = pickle.load(f)
else:
    print("File not found:", save_path)

In [ ]:
# assemble X_train matrix
X_train = []

# store contents of data file into q_r, o, a
for sequence in data:
    q_r, o, a = sequence

    print(q_r.shape)

    # for i in range(q_r.shape[0]):
        # vector_to_tile = np.append(q_r,o)
        # print(vector_to_tile)
        # X_t = np.tile(vector_to_tile[i,:],(num_particles,1))
        # X_train.append(X_t)

# X_train = np.vstack(X_train)

In [ ]:
# check that models have been built
dpf.build_networks()
print(dpf.obs_like_estimator)
print(dpf.particle_proposer)